In [ ]:
# imports and config
from pathlib import Path
import torch

from scripts.common import deep_update, load_config, print_config_summary
from scripts.train_cli import (
    DEFAULT_CONFIG,
    resolve_device_and_dtype,
    build_data,
    build_model_bundles,
    build_losses,
)
from src.training.train_aging_model import train_global_local_face_aging

config = deep_update(DEFAULT_CONFIG, load_config("configs/training/default_train.yaml"))

# Main editable hyperparameters
config["run"]["name"] = "notebook_global_local_run"
config["run"]["checkpoint_root"] = "training_checkpoints/notebook_run"

config["data"]["batch_size"] = 4
config["data"]["num_workers"] = 0
config["data"]["pin_memory"] = False

config["training"]["num_epochs"] = 5
config["training"]["train_order"] = ["local", "global"]
config["training"]["local_grad_accum_steps"] = 4
config["training"]["global_grad_accum_steps"] = 4

# Use these for a smoke run before real training
# config["training"]["local_max_batches"] = 1
# config["training"]["global_max_batches"] = 1

print_config_summary(config)

In [ ]:
from pathlib import Path
import data.global_path_datasets as global_paths
from data.create_data import build_global_dataloaders

# Carpeta donde quieres extraer/indexar las imágenes globales
global_paths.GLOBAL_IMAGE_DIR = Path("data/global_extracted")

# CSV con atributos/prompts/edad
global_paths.GLOBAL_CSV_PATH = Path("data/ffhq_predictions/ffhq_face_attribute_prompts.csv")

# Los 4 ZIP globales
global_paths.DRIVE_ZIPS = [
    Path("/ruta/a/ffhq_part_1.zip"),
    Path("/ruta/a/ffhq_part_2.zip"),
    Path("/ruta/a/ffhq_part_3.zip"),
    Path("/ruta/a/ffhq_part_4.zip"),
]


global_objects = build_global_dataloaders(
    batch_size=4,
    num_workers=0,
    pin_memory=False,
)

In [ ]:
# load device, dataloaders, diffusion bundles, and adapters
device, dtype = resolve_device_and_dtype(config)

# Builds:
# - local train/val loaders from data/data_subset + data/results_labeling
# - global train/val loaders from configured global paths
local_objects, _ = build_data(config)

# Loads:
# - global SD bundle
# - local SD bundle
# Then injects:
# - global LoRA
# - local DoRA
# And creates adapter optimizers
mixed_global_bundle, mixed_local_bundle = build_model_bundles(config, device, dtype)

In [ ]:
#  build ScoreNet and losses
# Uses config["score_net"]["checkpoint_path"].
# Local loss:
#   lambda_full, lambda_zone, lambda_score, lambda_cycle
# Global loss:
#   lambda_diff, lambda_id, lambda_age, lambda_delta_age, lambda_perc
local_loss, global_loss = build_losses(
    config=config,
    mixed_global_bundle=mixed_global_bundle,
    mixed_local_bundle=mixed_local_bundle,
    device=device,
)

In [ ]:
# train
train_cfg = config["training"]
sampling_cfg = config.get("sampling", {})

result = train_global_local_face_aging(
    mixed_local_bundle=mixed_local_bundle,
    mixed_global_bundle=mixed_global_bundle,
    local_train_loader=local_objects["train_loader"],
    global_train_loader=global_objects["train_loader"],
    local_loss_fn=local_loss,
    global_loss_fn=global_loss,
    device=device,
    amp_enabled=config["device"]["amp_enabled"],
    amp_dtype=config["device"]["amp_dtype"],
    run_name=config["run"]["name"],
    checkpoint_root=config["run"]["checkpoint_root"],
    sampling_loader_global=None,
    sampling_loader_local=None,
    sample_every_epochs=sampling_cfg.get("sample_every_epochs", 0),
    sampling_output_dir=sampling_cfg.get("sampling_output_dir"),
    **train_cfg,
)

result